
**🤖 AI Lab Partner Policy: STRICTLY Opt-In Code Generation**

In this course, we treat AI tools (like ChatGPT, Gemini, Copilot) as **Lab Partners**, not solution generators. You must use the following prompt to ensure the AI acts responsibly.

**1. Copy the text inside the block below**
**2. Open your AI Assistant (Gemini, ChatGPT, etc.)**
**3. Paste the text to set the rules for the session**

> "I am a student in an Intro to Machine Learning course. Please act as my **ML Lab Partner**.
> 
> **Your Rules:**
> 
> 1. **Code Generation is STRICTLY Opt-In:** You **MUST NOT** generate any runnable Python code unless my message starts with one of the specific prefixes below (`code:` or `output:`).
>    * *Default Behavior:* If I ask 'How do I...?' or 'Help me with...', explain the strategy in English, provide pseudocode, or use illustrative examples. Do not generate runnable solution code.
> 
> 2. **The 'code:' Trigger (Logic & Calculation):** 
>    * When generating code, prioritize simplicity and human readability. Avoid complex syntax.
>    * **Constraint:** When I use this trigger, provide **only one single line of code**. Do not write full blocks.
> 
> 3. **The 'output:' Trigger (Formatting & Printing):**
>    * Use this ONLY when I request code to print results, format tables, or create plots.
>    * **Exception:** For this trigger only, you **MAY** provide full multi-line code blocks to handle the verbose syntax of formatting or plotting.
> 
> 4. **Wait for Me:** After providing the code, stop immediately. Wait for me to run it and ask for the next step.
> 
> 5. **Explain Briefly:** Add a short comment explaining what the code does.
> 
> 6. **Catch Logic Errors:** If I ask for a step that is methodologically wrong (like testing on training data), stop me and explain the error before proceeding."


# Lecture 10: Multiple Regression & Least Squares

**Topics:**
1. Load Data and Train/Test Split
2. Simple Linear Regression (Review)
3. The Feature Matrix, Predictions, and Cost Function
4. Multiple Regression with sklearn
5. Train/Test Evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## 1. Load Education Dataset and Split into Train/Test

We have data on students' **math scores**, **reading scores**, and **socioeconomic status (SES)**.

**Goal:** Predict reading score from math score AND SES.

In [ ]:
# --- Load the Data ---
# Same ELS dataset as Lecture 8, now with SES (socioeconomic status)

df = pd.read_stata('https://raw.githubusercontent.com/fhfarnoud/intro2ml/main/2026S/data/els_small.dta')

# Clean: remove rows with non-numeric values
df = df[pd.to_numeric(df['bymath'], errors='coerce').notnull()]
df = df[pd.to_numeric(df['byread'], errors='coerce').notnull()]
df = df[pd.to_numeric(df['byses'], errors='coerce').notnull()]

# Extract as numpy arrays
mathscore = np.array(df['bymath'].values, dtype=float).reshape(-1, 1)
readscore = np.array(df['byread'].values, dtype=float).reshape(-1, 1)
ses = np.array(df['byses'].values, dtype=float).reshape(-1, 1)

# Split into train (80%) and test (20%)
math_train, math_test, read_train, read_test, ses_train, ses_test = train_test_split(
    mathscore, readscore, ses, test_size=0.2, random_state=42
)

print(f"Total students: {len(mathscore)}")
print(f"Training set: {len(math_train)}")
print(f"Test set: {len(math_test)}")

### Visualize the Data

Let's look at how reading score relates to each predictor.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Reading vs Math (training data only)
ax1.scatter(math_train, read_train, alpha=0.5)
ax1.set_xlabel('Math Score')
ax1.set_ylabel('Reading Score')
ax1.set_title('Reading vs Math Score (Training Data)')
ax1.grid(True, alpha=0.3)

# Reading vs SES (training data only)
ax2.scatter(ses_train, read_train, alpha=0.5, color='orange')
ax2.set_xlabel('Socioeconomic Status (SES)')
ax2.set_ylabel('Reading Score')
ax2.set_title('Reading vs SES (Training Data)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Simple Linear Regression (Review)

First, let's fit a model using **only math score**:
$$\text{reading} = a \cdot \text{math} + b$$

### Review: `LinearRegression` Syntax

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)           # X: features (N, D), y: targets (N,) or (N, 1)

model.coef_               # Coefficients (slopes)
model.intercept_          # Intercept (bias term)
```

In [ ]:
# YOUR CODE HERE
# TODO: Create and fit a LinearRegression model on math_train -> read_train
model_simple = ...
model_simple.fit(...)
#
# TODO: Extract the slope and intercept
a_simple = model_simple.coef_[...]
b_simple = model_simple.intercept_[...]
raise NotImplementedError()

print(f"Simple model: reading = {a_simple:.3f} * math + {b_simple:.3f}")

## 3. The Feature Matrix, Predictions, and Cost Function

In the slides, we introduced the **feature matrix** $X$, the prediction formula $\hat{\mathbf{y}} = X\mathbf{a}$, and the least-squares cost. Let's implement each step by hand.

### 3.1 Building the Feature Matrix

For multiple regression with intercept:
$$X = \begin{bmatrix} x_{1,1} & x_{1,2} & 1 \\ x_{2,1} & x_{2,2} & 1 \\ \vdots & \vdots & \vdots \\ x_{N,1} & x_{N,2} & 1 \end{bmatrix}$$

The column of 1s allows us to include the intercept in the matrix equation.

### NEW: `np.hstack()`

Horizontally stacks arrays (combines columns side by side):

```python
a = np.array([[1], [2], [3]])   # Shape: (3, 1)
b = np.array([[4], [5], [6]])   # Shape: (3, 1)
np.hstack((a, b))               # Shape: (3, 2) - columns side by side
# array([[1, 4],
#        [2, 5],
#        [3, 6]])
```

Create the feature matrix with columns: [math, SES, 1]

In [ ]:
# YOUR CODE HERE
# TODO: Create a column of ones with the same number of rows as training data
n_train = len(math_train)
ones_column = ...

# TODO: Build the feature matrix X_design with columns [math, SES, ones]
X_design = ...
raise NotImplementedError()

print(f"Feature matrix shape: {X_design.shape}")
print(f"First 5 rows of X_design:")
print(X_design[:5])

In [ ]:
"""Check"""
assert X_design.shape == (n_train, 3), f"X_design should have shape (n_train, 3), got {X_design.shape}"
assert np.all(X_design[:, 2] == 1), "Third column should be all 1s"
print("Check passed!")

### 3.2 Predictions with Matrix Multiplication

Once we have the coefficient vector $\mathbf{a} = [a_1, a_2, b]^T$, predictions are:
$$\hat{\mathbf{y}} = X\mathbf{a}$$

But first we need to find $\mathbf{a}$! The **normal equations** give the least-squares solution:

$$\hat{\mathbf{a}} = (X^TX)^{-1}X^T\mathbf{y}$$

### NEW: `.T` (Transpose)

Swaps rows and columns of an array:

```python
X = np.array([[1, 2, 3],
              [4, 5, 6]])   # Shape: (2, 3)
X.T                         # Shape: (3, 2)
# array([[1, 4],
#        [2, 5],
#        [3, 6]])
```

### NEW: `np.linalg.inv()`

Computes the inverse of a square matrix:

```python
A = np.array([[2, 0],
              [0, 3]])
A_inv = np.linalg.inv(A)    # Returns [[0.5, 0], [0, 0.333...]]
A @ A_inv                    # Returns identity matrix [[1, 0], [0, 1]]
```

In [ ]:
# YOUR CODE HERE
# TODO: Compute X^T X, its inverse, and X^T y
XtX = ...
XtX_inv = ...
Xty = ...
#
# TODO: Compute the coefficient vector a using the normal equations
a = ...
raise NotImplementedError()

# Extract individual coefficients
a1 = a[0, 0]
a2 = a[1, 0]
b = a[2, 0]

print("Normal equations solution:")
print(f"  a1 (math coef) = {a1:.4f}")
print(f"  a2 (SES coef)  = {a2:.4f}")
print(f"  b  (intercept) = {b:.4f}")

Now compute predictions using matrix multiplication: $\hat{\mathbf{y}} = X\mathbf{a}$

In [ ]:
# YOUR CODE HERE
# TODO: Compute predictions using X_design @ a
y_pred_train = ...
raise NotImplementedError()

print(f"First 5 predictions: {y_pred_train[:5].flatten()}")

# Verify: this should match the explicit formula a1*math + a2*ses + b
y_pred_check = a1 * math_train + a2 * ses_train + b
print(f"Matches explicit formula? {np.allclose(y_pred_train, y_pred_check)}")

### 3.3 Cost Function (SSE)

The Sum of Squared Errors in matrix form:
$$\text{SSE}(\mathbf{a}) = \| \mathbf{y} - X\mathbf{a} \|_2^2 = \langle \mathbf{y} - \hat{\mathbf{y}}, \mathbf{y} - \hat{\mathbf{y}} \rangle$$

Compute SSE for both models (simple vs multiple) and see which is lower.

In [ ]:
# YOUR CODE HERE
# TODO: Compute predictions for simple model (math only)
y_pred_simple = ...

# TODO: Compute SSE for simple model: sum of squared errors
sse_simple = ...

# TODO: Compute SSE for multiple model
sse_multi = ...
raise NotImplementedError()

print(f"SSE (simple model, math only): {sse_simple:.2f}")
print(f"SSE (multiple model, math+SES): {sse_multi:.2f}")
print(f"\nReduction in SSE: {sse_simple - sse_multi:.2f} ({100*(sse_simple-sse_multi)/sse_simple:.1f}%)")

**Question:** Adding SES reduced SSE by a relatively small amount. Why?

**Answer:**

## 4. Multiple Regression with sklearn

We just solved the normal equations by hand. `LinearRegression()` from sklearn does the same math automatically.

It doesn't need a separate mode for simple vs. multiple regression — it just looks at the **number of columns** in the input `X`:
- Section 2: `math_train` has shape `(N, 1)` — one column → one coefficient
- Now: we'll stack math and SES into shape `(N, 2)` — two columns → two coefficients

The `.fit(X, y)` call is identical either way.

Combine the two features into a single matrix (using `np.hstack`) and fit a `LinearRegression` model.

In [ ]:
# YOUR CODE HERE
# TODO: Create feature matrix X_train by combining math_train and ses_train columns
X_train = ...

# TODO: Fit a LinearRegression model
model_multi = ...
model_multi.fit(...)
raise NotImplementedError()

# Extract coefficients
a1, a2 = model_multi.coef_[0]
b = model_multi.intercept_[0]

print(f"Multiple model: reading = {a1:.3f} * math + {a2:.3f} * SES + {b:.3f}")

In [ ]:
"""Check"""
assert X_train.shape == (len(math_train), 2), f"X_train should have shape (n_train, 2), got {X_train.shape}"
print("Check passed!")

**Observation:** sklearn gives the same coefficients as our normal equations solution — it's doing the same math behind the scenes!

## 5. Train/Test Evaluation

We've been computing SSE on training data, but the real test is: **how well do the models predict on unseen data?**

Let's evaluate both models on the test set using RMSE.

### Review: RMSE

```python
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
```

In [ ]:
# Create test feature matrix
X_test = np.hstack((math_test, ses_test))

# YOUR CODE HERE
# TODO: Compute predictions on test data for both models
y_pred_simple_test = model_simple.predict(...)
y_pred_multi_test = model_multi.predict(...)
raise NotImplementedError()

print(f"Test set size: {len(math_test)}")

In [ ]:
# Compute predictions on training data for comparison
y_pred_simple_train = model_simple.predict(math_train)
y_pred_multi_train = model_multi.predict(X_train)

# YOUR CODE HERE
# TODO: Compute RMSE for simple model on train and test
rmse_simple_train = np.sqrt(mean_squared_error(...))
rmse_simple_test = np.sqrt(mean_squared_error(...))
#
# TODO: Compute RMSE for multiple model on train and test
rmse_multi_train = np.sqrt(mean_squared_error(...))
rmse_multi_test = np.sqrt(mean_squared_error(...))
raise NotImplementedError()

print("RMSE Results:")
print(f"{'Model':<20} {'Train RMSE':<15} {'Test RMSE':<15}")
print("-" * 50)
print(f"{'Simple (math only)':<20} {rmse_simple_train:<15.3f} {rmse_simple_test:<15.3f}")
print(f"{'Multiple (math+SES)':<20} {rmse_multi_train:<15.3f} {rmse_multi_test:<15.3f}")

**Question:** Does adding SES improve predictions on the test set?

**Answer:**

---
## Summary

In this notebook, we learned:

1. **Multiple Linear Regression** extends simple regression to multiple input variables
2. The **Feature Matrix** $X$ organizes features and includes a column of 1s for the intercept
3. **Predictions** can be computed as $\hat{\mathbf{y}} = X\mathbf{a}$
4. The **Normal Equations** give the closed-form solution: $\hat{\mathbf{a}} = (X^TX)^{-1}X^T\mathbf{y}$
5. **Train/Test Split** lets us evaluate how well models generalize to unseen data
6. Adding more predictors can reduce error, but the benefit should be verified on test data

**Next time:** Polynomial regression and overfitting.